In [1]:
# Install required packages
!pip install boto3 pandas numpy scikit-learn sentence-transformers datarec-lib requests

  Using cached boto3-1.43.2-py3-none-any.whl.metadata (6.5 kB)
  Using cached sentence_transformers-5.4.1-py3-none-any.whl.metadata (17 kB)
  Using cached datarec_lib-1.5.7-py3-none-any.whl.metadata (1.3 kB)
  Using cached botocore-1.43.2-py3-none-any.whl.metadata (5.5 kB)
  Using cached jmespath-1.1.0-py3-none-any.whl.metadata (7.6 kB)
  Using cached s3transfer-0.17.0-py3-none-any.whl.metadata (1.7 kB)
  Using cached transformers-5.7.0-py3-none-any.whl.metadata (33 kB)
  Using cached huggingface_hub-1.13.0-py3-none-any.whl.metadata (14 kB)
  Using cached torch-2.11.0-cp311-cp311-manylinux_2_28_x86_64.whl.metadata (29 kB)
  Using cached datarec-1.5.7-py3-none-any.whl.metadata (10 kB)
  Using cached pandas-2.3.3-cp311-cp311-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl.metadata (91 kB)
  Using cached scikit_learn-1.8.0-cp311-cp311-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (11 kB)
  Using cached gdown-4.7.3-py3-none-any.whl.metadata (4.4 kB)
  Using cached kagglehub-1.0.

In [2]:
# Importing needed packages
import os
import zipfile
import pandas as pd
import requests
import numpy as np
import requests
from io import BytesIO

import boto3
from botocore import UNSIGNED
from botocore.config import Config
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

from transformers import BertTokenizer, BertModel
import torch

In [ ]:
# Task 1 - Check if files exist
BUCKET = "ynf0058-de300-lab3"
KEY = "ml-1m/ratings.dat" 

s3 = boto3.client(
    "s3",
    region_name="us-east-1",
    config=Config(signature_version=UNSIGNED)
)

def file_exists_public(bucket, key):
    try:
        s3.head_object(Bucket=bucket, Key=key)
        return True
    except:
        return False

print(file_exists_public(BUCKET, KEY))

True


In [4]:
# Task 2 - Creating the embeddings
## Starting with reading in the dataset
url_for_movies = "https://ynf0058-de300-lab3.s3.amazonaws.com/ml-1m/movies.dat"
def load_movies():
    movies = pd.read_csv(url_for_movies, sep= "::", engine = "python", names = ["MovieID", "Title", "Genres"], encoding = "latin-1")
    return movies

movies = load_movies()

In [5]:
# Next, we determine the year of movie release by extracting it from the Title column and filtering out those released pre 1980.
def filter_movie_year(movies):
    movies = movies.copy()
    movies["Year"] = movies["Title"].str.extract(r"\((\d{4})\)").astype(float)
    movies = movies[movies["Year"] <= 1980].copy()
    return movies

movies_pre_1980 = filter_movie_year(movies)
movies_pre_1980.head()

,MovieID,Title,Genres,Year
109,111,Taxi Driver (1976),Drama|Thriller,1976.0
152,154,Belle de jour (1967),Drama,1967.0
197,199,"Umbrellas of Cherbourg, The (Parapluies de Che...",Drama|Musical,1964.0
257,260,Star Wars: Episode IV - A New Hope (1977),Action|Adventure|Fantasy|Sci-Fi,1977.0
386,390,Faster Pussycat! Kill! Kill! (1965),Action|Comedy|Drama,1965.0


In [ ]:
# Creates the specific text to be used by the BERT algorithm
def create_bert_text(movies):
    movies = movies.copy()
    movies["bert_text"] = ("Movie title: " + movies["Title"] + ". Genres: " + movies["Genres"])
    return movies

movies_pre_1980 = create_bert_text(movies_pre_1980)
movies_pre_1980[["MovieID", "Title", "Genres", "Year", "bert_text"]].head()

,MovieID,Title,Genres,Year,bert_text
109,111,Taxi Driver (1976),Drama|Thriller,1976.0,Movie title: Taxi Driver (1976). Genres: Drama...
152,154,Belle de jour (1967),Drama,1967.0,Movie title: Belle de jour (1967). Genres: Drama
197,199,"Umbrellas of Cherbourg, The (Parapluies de Che...",Drama|Musical,1964.0,"Movie title: Umbrellas of Cherbourg, The (Para..."
257,260,Star Wars: Episode IV - A New Hope (1977),Action|Adventure|Fantasy|Sci-Fi,1977.0,Movie title: Star Wars: Episode IV - A New Hop...
386,390,Faster Pussycat! Kill! Kill! (1965),Action|Comedy|Drama,1965.0,Movie title: Faster Pussycat! Kill! Kill! (196...


In [ ]:
# Uses BERT to generate embeddings from the BERT text
def create_raw_bert_embeddings(movies):
    tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
    model = BertModel.from_pretrained("bert-base-uncased")
    model.eval()
    texts = movies["bert_text"].tolist()
    embeddings = []
    for text in texts:
        inputs = tokenizer(text, return_tensors = "pt", truncation = True, max_length = 128)
        with torch.no_grad():
            outputs = model(**inputs)

        cls_embedding = outputs.last_hidden_state[:, 0, :]
        embeddings.append(cls_embedding.squeeze().numpy())

    return np.array(embeddings)

embeddings_pre_1980 = create_raw_bert_embeddings(movies_pre_1980)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [8]:
# Saving Task 2 files
os.makedirs("outputs", exist_ok = True)
movies_pre_1980.to_csv("outputs/movies_pre_1980.csv", index = False)
np.save("outputs/movie_embeddings_pre_1980.npy", embeddings_pre_1980)

os.listdir("outputs")

['ec2_bert_smoketest.json',
 'movies_pre_1980.csv',
 'movie_embeddings_pre_1980.npy',
 '.ipynb_checkpoints']

In [9]:
# Task 3 - Loading intermediates from the S3 bucket
movies_url = f"https://{BUCKET}.s3.amazonaws.com/task2_processed/movies_pre_1980.csv"
embeddings_url = f"https://{BUCKET}.s3.amazonaws.com/task2_processed/movie_embeddings_pre_1980.npy"
movies_pre_1980 = pd.read_csv(movies_url)

response = requests.get(embeddings_url)
embeddings = np.load(BytesIO(response.content))

print(movies_pre_1980.shape)
print(embeddings.shape)


(887, 5)
(887, 768)


In [10]:
# Loading the ratings file
ratings_url = f"https://{BUCKET}.s3.amazonaws.com/ml-1m/ratings.dat"
ratings = pd.read_csv(ratings_url, sep = "::", engine = "python", names = ["UserID", "MovieID", "Rating", "Timestamp"], encoding = "latin-1")

ratings.head()

,UserID,MovieID,Rating,Timestamp
0,1,1193,5,978300760
1,1,661,3,978302109
2,1,914,3,978301968
3,1,3408,4,978300275
4,1,2355,5,978824291


In [11]:
# Defining a cold user's recommendations
def cold_user_recommendation(movies, embeddings, top_k = 5):
    average_embedding = embeddings.mean(axis = 0).reshape(1, -1)
    similarity = cosine_similarity(average_embedding, embeddings)[0]
    top_indices = similarity.argsort()[-top_k:][::-1]
    recs = movies.iloc[top_indices][["MovieID", "Title", "Genres", "Year"]].copy()
    recs["Similarity"] = similarity[top_indices]
    return recs

cold_recs = cold_user_recommendation(movies_pre_1980, embeddings)
cold_recs

,MovieID,Title,Genres,Year,Similarity
715,3461,Lord of the Flies (1963),Adventure|Drama|Thriller,1963.0,0.984877
78,939,"Reluctant Debutante, The (1958)",Comedy|Drama,1958.0,0.984798
343,2160,Rosemary's Baby (1968),Horror|Thriller,1968.0,0.984258
718,3468,"Hustler, The (1961)",Drama,1961.0,0.984221
282,1953,"French Connection, The (1971)",Action|Crime|Drama|Thriller,1971.0,0.983282


In [12]:
# Selecting the top user
def select_top_user(ratings):
    count_per_user = ratings.groupby("UserID").size().reset_index(name = "Interaction_Count")
    upper_threshold = count_per_user["Interaction_Count"].quantile(0.95)
    top_users = count_per_user[count_per_user["Interaction_Count"] >= upper_threshold]
    chosen_user = top_users.sample(1, random_state = 42).iloc[0]
    return int(chosen_user["UserID"]), int(chosen_user["Interaction_Count"])
top_user_id, top_user_interactions = select_top_user(ratings)
top_user_id, top_user_interactions

(3519, 666)

In [13]:
# Defining recommendations for the top user
def top_user_recommendation(user_id, ratings, movies, embeddings, top_k = 5, min_rating = 4):
    movie_id_to_index = {movie_id: i for i, movie_id in enumerate(movies["MovieID"])}
    user_ratings = ratings[(ratings["UserID"] == user_id) & (ratings["Rating"] >= min_rating)].copy()
    user_ratings = user_ratings[user_ratings["MovieID"].isin(movie_id_to_index.keys())]
    rated_indices = [movie_id_to_index[movie_id] for movie_id in user_ratings["MovieID"]]
    user_embedding = embeddings[rated_indices].mean(axis = 0).reshape(1, -1)
    similarity = cosine_similarity(user_embedding, embeddings)[0]
    already_rated = set(user_ratings["MovieID"])
    candidate_indices = [i for i, movie_id in enumerate(movies["MovieID"]) if movie_id not in already_rated]
    ranked_indices = sorted(candidate_indices, key = lambda i: similarity[i], reverse = True)
    top_indices = ranked_indices[: top_k]
    recs = movies.iloc[top_indices][["MovieID", "Title", "Genres", "Year"]].copy()
    recs["Similarity"] = similarity[top_indices]
    return recs

top_user_recs = top_user_recommendation(top_user_id, ratings, movies_pre_1980, embeddings)
top_user_recs

,MovieID,Title,Genres,Year,Similarity
243,1371,Star Trek: The Motion Picture (1979),Action|Adventure|Sci-Fi,1979.0,0.987601
501,2851,Saturn 3 (1979),Adventure|Sci-Fi|Thriller,1979.0,0.987265
283,1954,Rocky (1976),Action|Drama,1976.0,0.986907
409,2409,Rocky II (1979),Action|Drama,1979.0,0.985884
858,3834,Bronco Billy (1980),Adventure|Drama|Romance,1980.0,0.985684


In [14]:
# Saving results as a csv
def create_task3_results(cold_recs, top_user_recs, top_user_id, top_user_interactions, ratings):
    top_user_data = ratings[ratings["UserID"] == top_user_id]
    last_interaction_time = top_user_data["Timestamp"].max()
    avg_rating = top_user_data["Rating"].mean()
    rows = []
    for _, row in cold_recs.iterrows():
        rows.append({
            "User_Type": "Cold User", 
            "User_ID": None, 
            "Last_Interaction_Time": None, 
            "User_Summary": "No prior history of rating", 
            "Recommended_MovieID": row["MovieID"],
            "Recommended_Title": row["Title"], 
            "Recommended_Genres": row["Genres"], 
            "Similarity": row["Similarity"]})
    for _, row in top_user_recs.iterrows():
        rows.append({
            "User_Type": "Top User", 
            "User_ID": top_user_id, 
            "Last_Interaction_Time": last_interaction_time, 
            "User_Summary": f"Top 5% user with {top_user_interactions} interactions and average rating {avg_rating:.2f}.", 
            "Recommended_MovieID": row["MovieID"],
            "Recommended_Title": row["Title"], 
            "Recommended_Genres": row["Genres"], 
            "Similarity": row["Similarity"]})
    return pd.DataFrame (rows)

task3_results = create_task3_results(cold_recs, top_user_recs, top_user_id, top_user_interactions, ratings)
task3_results





,User_Type,User_ID,Last_Interaction_Time,User_Summary,Recommended_MovieID,Recommended_Title,Recommended_Genres,Similarity
0,Cold User,NaN,NaN,No prior history of rating,3461,Lord of the Flies (1963),Adventure|Drama|Thriller,0.984877
1,Cold User,NaN,NaN,No prior history of rating,939,"Reluctant Debutante, The (1958)",Comedy|Drama,0.984798
2,Cold User,NaN,NaN,No prior history of rating,2160,Rosemary's Baby (1968),Horror|Thriller,0.984258
3,Cold User,NaN,NaN,No prior history of rating,3468,"Hustler, The (1961)",Drama,0.984221
4,Cold User,NaN,NaN,No prior history of rating,1953,"French Connection, The (1971)",Action|Crime|Drama|Thriller,0.983282
5,Top User,3519.0,1.045349e+09,Top 5% user with 666 interactions and average ...,1371,Star Trek: The Motion Picture (1979),Action|Adventure|Sci-Fi,0.987601
6,Top User,3519.0,1.045349e+09,Top 5% user with 666 interactions and average ...,2851,Saturn 3 (1979),Adventure|Sci-Fi|Thriller,0.987265
7,Top User,3519.0,1.045349e+09,Top 5% user with 666 interactions and average ...,1954,Rocky (1976),Action|Drama,0.986907
8,Top User,3519.0,1.045349e+09,Top 5% user with 666 interactions and average ...,2409,Rocky II (1979),Action|Drama,0.985884
9,Top User,3519.0,1.045349e+09,Top 5% user with 666 interactions and average ...,3834,Bronco Billy (1980),Adventure|Drama|Romance,0.985684


In [15]:
# Saving file to outputs
task3_results.to_csv("outputs/task3_recommendations_pre1980.csv", index = False)

In [17]:
# Task 4 - Reusing functions from Tasks 2 and 3 for the entire dataset
# First, loading the dataset in:
full_movies = load_movies()
full_movies = create_bert_text(full_movies)

full_movies.head()

,MovieID,Title,Genres,bert_text
0,1,Toy Story (1995),Animation|Children's|Comedy,Movie title: Toy Story (1995). Genres: Animati...
1,2,Jumanji (1995),Adventure|Children's|Fantasy,Movie title: Jumanji (1995). Genres: Adventure...
2,3,Grumpier Old Men (1995),Comedy|Romance,Movie title: Grumpier Old Men (1995). Genres: ...
3,4,Waiting to Exhale (1995),Comedy|Drama,Movie title: Waiting to Exhale (1995). Genres:...
4,5,Father of the Bride Part II (1995),Comedy,Movie title: Father of the Bride Part II (1995...


In [18]:
# Creating BERT embeddings for all movies
full_embeddings = create_raw_bert_embeddings(full_movies)
print(full_movies.shape)
print(full_embeddings.shape)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


(3883, 4)
(3883, 768)


In [19]:
# Savings files locally
full_movies.to_csv("outputs/movies_full.csv", index = False)
np.save("outputs/movie_embeddings_full", full_embeddings)

In [21]:
# Loading files from S3 bucket again (for robustness)
full_movies_url = f"https://{BUCKET}.s3.amazonaws.com/task4_full/movies_full.csv"
full_embeddings_url = f"https://{BUCKET}.s3.amazonaws.com/task4_full/movie_embeddings_full.npy"
full_movies = pd.read_csv(full_movies_url)
response = requests.get(full_embeddings_url)
full_embeddings = np.load(BytesIO(response.content))
print(full_movies.shape)
print(full_embeddings.shape)

(3883, 4)
(3883, 768)


In [23]:
# Adding a "Year" column to the full dataset for consistency in function application
def add_year(movies):
    movies = movies.copy()
    movies["Year"] = movies["Title"].str.extract(r"\((\d{4})\)").astype(float)
    return movies

full_movies = add_year(full_movies)

In [24]:
# Getting the recommendations for the cold users
cold_recs_full = cold_user_recommendation(full_movies, full_embeddings)
cold_recs_full

,MovieID,Title,Genres,Year,Similarity
1494,1531,Losing Chase (1996),Drama,1996.0,0.987940
1518,1557,Squeeze (1996),Drama,1996.0,0.987569
3,4,Waiting to Exhale (1995),Comedy|Drama,1995.0,0.987481
1259,1279,Night on Earth (1991),Comedy|Drama,1991.0,0.987048
1027,1040,"Secret Agent, The (1996)",Drama,1996.0,0.986575


In [25]:
top_user_recs_full = top_user_recommendation(top_user_id, ratings, full_movies, full_embeddings)
top_user_recs_full

,MovieID,Title,Genres,Year,Similarity
3,4,Waiting to Exhale (1995),Comedy|Drama,1995.0,0.990988
2857,2926,Hairspray (1988),Comedy|Drama,1988.0,0.989094
1525,1565,Head Above Water (1996),Comedy|Thriller,1996.0,0.988118
1792,1861,Junk Mail (1997),Comedy|Thriller,1997.0,0.987968
1579,1622,Kicked in the Head (1997),Comedy|Drama,1997.0,0.987931


In [26]:
# Saving task 4 file
task4_results = create_task3_results(cold_recs_full, top_user_recs_full, top_user_id, top_user_interactions, ratings)
task4_results.to_csv("outputs/task4_recommendations_full.csv", index = False)
task4_results

,User_Type,User_ID,Last_Interaction_Time,User_Summary,Recommended_MovieID,Recommended_Title,Recommended_Genres,Similarity
0,Cold User,NaN,NaN,No prior history of rating,1531,Losing Chase (1996),Drama,0.987940
1,Cold User,NaN,NaN,No prior history of rating,1557,Squeeze (1996),Drama,0.987569
2,Cold User,NaN,NaN,No prior history of rating,4,Waiting to Exhale (1995),Comedy|Drama,0.987481
3,Cold User,NaN,NaN,No prior history of rating,1279,Night on Earth (1991),Comedy|Drama,0.987048
4,Cold User,NaN,NaN,No prior history of rating,1040,"Secret Agent, The (1996)",Drama,0.986575
5,Top User,3519.0,1.045349e+09,Top 5% user with 666 interactions and average ...,4,Waiting to Exhale (1995),Comedy|Drama,0.990988
6,Top User,3519.0,1.045349e+09,Top 5% user with 666 interactions and average ...,2926,Hairspray (1988),Comedy|Drama,0.989094
7,Top User,3519.0,1.045349e+09,Top 5% user with 666 interactions and average ...,1565,Head Above Water (1996),Comedy|Thriller,0.988118
8,Top User,3519.0,1.045349e+09,Top 5% user with 666 interactions and average ...,1861,Junk Mail (1997),Comedy|Thriller,0.987968
9,Top User,3519.0,1.045349e+09,Top 5% user with 666 interactions and average ...,1622,Kicked in the Head (1997),Comedy|Drama,0.987931


In [28]:
# Task 5 - Creating a personal user profile
my_preferences = pd.DataFrame({"MovieID": [1, 2, 48, 57, 70, 104, 147, 153, 165, 223], "Rating": [5, 4, 3, 5, 3, 5, 4, 5, 5, 3]})
my_user_profile = my_preferences.merge(full_movies[["MovieID", "Title", "Genres", "Year"]], on = "MovieID", how = "left")
my_user_profile
        

,MovieID,Rating,Title,Genres,Year
0,1,5,Toy Story (1995),Animation|Children's|Comedy,1995.0
1,2,4,Jumanji (1995),Adventure|Children's|Fantasy,1995.0
2,48,3,Pocahontas (1995),Animation|Children's|Musical|Romance,1995.0
3,57,5,Home for the Holidays (1995),Drama,1995.0
4,70,3,From Dusk Till Dawn (1996),Action|Comedy|Crime|Horror|Thriller,1996.0
5,104,5,Happy Gilmore (1996),Comedy,1996.0
6,147,4,"Basketball Diaries, The (1995)",Drama,1995.0
7,153,5,Batman Forever (1995),Action|Adventure|Comedy|Crime,1995.0
8,165,5,Die Hard: With a Vengeance (1995),Action|Thriller,1995.0
9,223,3,Clerks (1994),Comedy,1994.0


In [32]:
# Recommending movies for myself
def recommend_for_me(user_profile, movies, embeddings, top_k = 5):
    movie_id_to_index = {movie_id: i for i, movie_id in enumerate(movies["MovieID"])}
    liked_movies = user_profile[user_profile["Rating"] >= 4]
    liked_indices = [movie_id_to_index[mid] for mid in liked_movies["MovieID"] if mid in movie_id_to_index]
    user_embedding = embeddings[liked_indices].mean(axis = 0).reshape(1, -1)
    similarity = cosine_similarity(user_embedding, embeddings)[0]
    already_rated = set(user_profile["MovieID"])
    candidate_indices = [i for i, mid in enumerate(movies["MovieID"]) if mid not in already_rated]
    ranked_indices = sorted(candidate_indices, key = lambda i: similarity[i], reverse = True)
    top_indices = ranked_indices[:top_k]
    recs = movies.iloc[top_indices][["MovieID", "Title", "Genres", "Year"]].copy()
    recs["Similarity"] = similarity[top_indices]
    return recs

my_recs = recommend_for_me(my_user_profile, full_movies, full_embeddings)
my_recs

,MovieID,Title,Genres,Year,Similarity
3,4,Waiting to Exhale (1995),Comedy|Drama,1995.0,0.988272
558,562,Welcome to the Dollhouse (1995),Comedy|Drama,1995.0,0.987541
1259,1279,Night on Earth (1991),Comedy|Drama,1991.0,0.986179
2857,2926,Hairspray (1988),Comedy|Drama,1988.0,0.986079
1099,1115,Sleepover (1995),Comedy|Drama,1995.0,0.985912


In [34]:
# Saving files
my_user_profile.to_csv("outputs/my_user_profile.csv", index = False)
my_recs.to_csv("outputs/my_recommendations.csv", index = False)